In [15]:
!pip install optuna

In [16]:
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from scipy.spatial import distance
import numpy as np
import optuna

In [17]:
# Load the Iris dataset
iris = load_iris()
X = iris.data

In [18]:
# Define objective function for Optuna
def objective(trial):
    # Choose algorithm
    algo = trial.suggest_categorical("algorithm", ["kmeans", "dbscan", "gmm"])
    
    if algo == "kmeans":
        n_clusters = trial.suggest_int("n_clusters", 2, 10)
        model = KMeans(n_clusters=n_clusters, random_state=42)
    
    elif algo == "dbscan":
        eps = trial.suggest_float("eps", 0.1, 5.0, log=True)
        min_samples = trial.suggest_int("min_samples", 2, 20)
        model = DBSCAN(eps=eps, min_samples=min_samples)
    
    else:  # gmm
        n_components = trial.suggest_int("n_components", 2, 10)
        covariance_type = trial.suggest_categorical("covariance_type", 
                                                    ["full", "tied", "diag", "spherical"])
        model = GaussianMixture(n_components=n_components, 
                                covariance_type=covariance_type,
                                random_state=42)

    # Fit model
    labels = model.fit_predict(X) if algo != "gmm" else model.fit(X).predict(X)

    # Handle case where clustering fails (all in one cluster or noise)
    if len(set(labels)) <= 1:
        return -1.0
    
    # Evaluate with silhouette score
    score = silhouette_score(X, labels)
    return score

In [19]:
# Run Optuna study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Best trial:")
print(study.best_trial.params)

[I 2025-08-21 14:44:52,089] A new study created in memory with name: no-name-5da291a6-ab2a-47af-be99-6aede3b65703
[I 2025-08-21 14:44:52,225] Trial 0 finished with value: 0.26577110648690666 and parameters: {'algorithm': 'gmm', 'n_components': 8, 'covariance_type': 'full'}. Best is trial 0 with value: 0.26577110648690666.
/opt/conda/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
[I 2025-08-21 14:44:52,351] Trial 1 finished with value: 0.6810461692117462 and parameters: {'algorithm': 'kmeans', 'n_clusters': 2}. Best is trial 1 with value: 0.6810461692117462.
[I 2025-08-21 14:44:52,359] Trial 2 finished with value: -1.0 and parameters: {'algorithm': 'dbscan', 'eps': 1.8943595552527475, 'min_samples': 13}. Best is trial 1 with value: 0.6810461692117462.
[I 2025-08-21 14:44:52,365] Trial 3 finished with value: -1.0 an

Best trial:
{'algorithm': 'gmm', 'n_components': 2, 'covariance_type': 'tied'}


In [20]:
# Retrain best model
best_params = study.best_trial.params
algo = best_params["algorithm"]

if algo == "kmeans":
    model = KMeans(n_clusters=best_params["n_clusters"], random_state=42)

elif algo == "dbscan":
    model = DBSCAN(eps=best_params["eps"], min_samples=best_params["min_samples"])

else:  # gmm
    model = GaussianMixture(n_components=best_params["n_components"], 
                            covariance_type=best_params["covariance_type"],
                            random_state=42)

In [21]:
labels = model.fit_predict(X) if algo != "gmm" else model.fit(X).predict(X)

print(f"Final silhouette score: {silhouette_score(X, labels):.4f}")

Final silhouette score: 0.6867
